
# <font color="green">Inner product with SIMD (beating the compiler)</font>

## Problem

* Write a function `inner_prod_simd(p, q, n)` __in assembly__ that computes the inner (dot) product of two `n`-element `double` arrays, using NEON to process more than one element per iteration.
* That is, compute the same value as this C function, but **faster than `gcc -O3`**:
```
double inner_prod_simd(double * p, double * q, long n) {
  double s = 0.0;
  for (long i = 0; i < n; i++) s += p[i] * q[i];
  return s;
}
```
* Suggested approach:
  1. zero a vector accumulator `v0.2d`;
  2. loop while at least 2 elements remain:
     `ld1 {v1.2d}, [x0], #16` ; `ld1 {v2.2d}, [x1], #16` ; `fmla v0.2d, v1.2d, v2.2d`;
  3. after the loop, reduce with `faddp d0, v0.2d`;
  4. handle the **tail**: if `n` is odd, add `p[last]*q[last]`.
* For extra speedup, use two or four vector accumulators and combine them at the end.
* Fill in the skeleton `inner_prod_simd.s` (after `// ------- write your answer here -------`).
* The checker `check_inner_prod_simd.c` verifies your result against a scalar reference (within a small tolerance, since the rounding may differ) and prints a timing comparison. If you see `OK` and a `speedup` greater than 1, you have beaten the compiler.

## Hints

* This is the same computation as *Inner product (dot product)*, but the goal is to be **faster than `gcc -O3`** by doing something the compiler is not allowed to do.
* Compile the scalar version (in the *Observe* cells below as `inner_prod_scalar`) and you will see a **serial chain** accumulating into a single register. The compiler keeps the additions serial because floating-point addition is **not associative**: reordering them would change the rounding. Without `-ffast-math`, it will not turn this into a parallel (lane-wise) reduction.
* But *you* are allowed to accept a slightly different rounding. So you can:
  * use **SIMD (NEON)** to multiply-accumulate several elements at once, and
  * use **several independent accumulators** to break the loop-carried dependency chain.
* A few NEON instructions:
  * `v0`–`v31` are 128-bit vector registers; as `.2d` they hold **two** `double`s.
  * `ld1 {v1.2d}, [x0], #16` --- load two `double`s and advance `x0` by 16 (post-increment).
  * `fmla v0.2d, v1.2d, v2.2d` --- fused multiply-add, lane-wise: `v0[i] += v1[i] * v2[i]`.
  * `faddp d0, v0.2d` --- horizontal add of the two lanes (use once at the end).
  * `movi v0.2d, #0` --- zero a vector accumulator.
* The *Observe* cells also contain `vmul` (`c[i] = p[i]*q[i]`), a loop the compiler **will** auto-vectorize, because each output is independent --- a useful contrast with the reduction.
* This is an **optional, advanced** problem. The point is conceptual: the compiler is conservative because it must preserve the exact floating-point rounding; when you know a slightly different rounding is acceptable, you can do better by hand.



# 1. AI Tutor
## 1-1. Prepare
* Your personal AI tutor is provided for questions and feedback.
* Execute the following cell before you use it.

In [ ]:
import heytutor

## 1-2. Examples
* A general question
```
%%hey
What does the `ldr` instruction do in ARM64?
```

* A hint on this specific problem
```
%%hey problem_file=inner_prod_simd.md
Give me a hint on this problem.

{problem}
```

* Builtin variables usable in `%%hey` cells
  * `{file:FILENAME}` is the content of FILE
  * `{bash[-1]}` is the output of the last `%%bash_` cell, `{bash[-2]}` the second last, etc.
  * `{problem}` is the content of the file you specify by `%%hey problem_file=foo.md`
  * `{answer}` is the content of the file you specify by `%%hey answer_file=foo.s`


# 2. Observe: compile example functions
* Before writing your own assembly, it helps to see what the compiler generates for related example functions.
* Running the first cell below writes `explore.c` (some small example functions related to this problem).
* The second cell compiles it with `gcc -O3 -S` and prints the generated assembly.
* Feel free to edit `explore.c` (change the code, add functions, change constants) and re-run the two cells to see how the assembly changes.

In [ ]:
%%writefile_ explore.c
/* A floating-point dot product. The compiler keeps the additions as a SERIAL
   chain of fmadd/fadd into a single accumulator (a loop-carried dependency):
   even if it uses vector LOADS, it does NOT add the lanes in parallel, because
   reordering floating-point additions would change the rounding of the result. */
double inner_prod_scalar(double * p, double * q, long n) {
  double s = 0.0;
  for (long i = 0; i < n; i++) s += p[i] * q[i];
  return s;
}

/* For contrast: this loop HAS no loop-carried fp dependency (each c[i] is
   independent), so the compiler WILL auto-vectorize it with v-registers. */
void vmul(double * p, double * q, double * c, long n) {
  for (long i = 0; i < n; i++) c[i] = p[i] * q[i];
}

In [ ]:
%%bash_
gcc -O3 -S explore.c
cat explore.s


# 3. Your Answer (assembly)
* Running the cell below writes the skeleton assembly file `inner_prod_simd.s`.
* Fill in your instructions after the line `// ------- write your answer here -------`, then run the cell again to save it.

In [ ]:
%%writefile_ inner_prod_simd.s
	.arch armv8-a
	.file	"inner_prod_simd.c"
	.text
	.align	2
	.p2align 4,,11
	.global	inner_prod_simd
	.type	inner_prod_simd, %function
inner_prod_simd:
.LFB0:
	.cfi_startproc
	// ------- write your answer here -------
	// hint: accumulate two products at a time in a v-register, e.g.
	//   ld1  {v1.2d}, [x0], #16     // p[i], p[i+1]; advance x0
	//   ld1  {v2.2d}, [x1], #16     // q[i], q[i+1]; advance x1
	//   fmla v0.2d, v1.2d, v2.2d    // v0 += p*q, lane-wise (fused multiply-add)
	// then reduce the two lanes at the end with:  faddp d0, v0.2d
	// don't forget the tail element when n is odd.
	.cfi_endproc
.LFE0:
	.size	inner_prod_simd, .-inner_prod_simd
	.ident	"GCC: (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0"
	.section	.note.GNU-stack,"",@progbits


# 4. Checker
* The following C program calls your `inner_prod_simd` function and checks the result against a reference computed in C.

In [ ]:
%%writefile_ check_inner_prod_simd.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>

double inner_prod_simd(double * p, double * q, long n);

/* scalar reference: a plain serial dot product (the compiler keeps it scalar). */
static double inner_prod_ref(double * p, double * q, long n) {
  double s = 0.0;
  for (long i = 0; i < n; i++) s += p[i] * q[i];
  return s;
}

int main(int argc, char ** argv) {
  long n = (argc >= 2) ? atol(argv[1]) : 1000000;
  double * p = (double *) malloc(n * sizeof(double));
  double * q = (double *) malloc(n * sizeof(double));
  for (long i = 0; i < n; i++) { p[i] = (double)((i % 100) + 1) * 0.5; q[i] = (double)((i % 13) + 1); }

  double r  = inner_prod_simd(p, q, n);
  double rc = inner_prod_ref(p, q, n);

  /* reordering the additions changes the rounding slightly, so allow a tolerance */
  double tol = 1e-6 * (1.0 + fabs(rc));
  int ok = (fabs(r - rc) <= tol);
  printf("%s dot: yours=%.6f ref=%.6f (|diff|=%.3g, tol=%.3g)\n",
         ok ? "OK" : "NG", r, rc, fabs(r - rc), tol);

  /* informational timing only (not part of pass/fail) */
  int reps = 200;
  volatile double sink = 0.0;
  clock_t t0 = clock();
  for (int k = 0; k < reps; k++) { p[0] = (double) k; sink += inner_prod_ref(p, q, n); }
  clock_t t1 = clock();
  for (int k = 0; k < reps; k++) { p[0] = (double) k; sink += inner_prod_simd(p, q, n); }
  clock_t t2 = clock();
  double ts = (double)(t1 - t0) / CLOCKS_PER_SEC;
  double ty = (double)(t2 - t1) / CLOCKS_PER_SEC;
  printf("timing over %d reps (n=%ld): scalar=%.3f s, yours=%.3f s, speedup=%.2fx\n",
         reps, n, ts, ty, ts / (ty > 1e-9 ? ty : 1e-9));

  free(p); free(q);
  return ok ? 0 : 1;
}


# 5. Compile
* Compile your assembly together with the checker.
* If you get an error, fix `inner_prod_simd.s` above and recompile.

In [ ]:
%%bash_
gcc -o check_inner_prod_simd -O3 check_inner_prod_simd.c inner_prod_simd.s -lm


# 6. Run
* If you see `OK`s and no errors, you are done.

In [ ]:
%%bash_
./check_inner_prod_simd 1000003


# 7. If things do not go well
* If your program compiles but does not produce the correct answer, run it within a debugger (gdb).
* Compile with `-O0 -g` first:
```
gcc -o check_inner_prod_simd -O0 -g check_inner_prod_simd.c inner_prod_simd.s -lm
```
* Then, in a terminal (SSH or Jupyter terminal):
```
gdb check_inner_prod_simd
(gdb) break inner_prod_simd
(gdb) run ...        # give the same arguments as above
```
* Step through one instruction at a time with `step`, and inspect registers with `print $x0` or `info registers`.

# 8. Ask Questions or Get Feedback
* You are encouraged to ask for feedback once you think you are done, to know if there is a better answer.

In [ ]:
%%hey problem_file=inner_prod_simd.md answer_file=inner_prod_simd.s

Problem:
{problem}

My Answer:
{answer}

Give me a feedback to my answer.